Проведем парсинг сайта Ostrovok.

Воспользуемся списком всех городов, получившихся при сборе наиболее релевантных.

In [ ]:
import pandas as pd
my_cities = pd.read_excel('Итоговый список городов с en.xlsx')
my_cities

,Unnamed: 0,Город,Регион,Федеральный округ,Население,Наличие аэропорта,name,name_en
0,0,Абакан,Хакасия,Сибирский,184284,True,Абакан,abakan
1,1,Анадырь,Чукотский АО,Дальневосточный,13224,True,Анадырь,anadyr
2,2,Анапа,Краснодарский край,Южный,84804,True,Анапа,anapa
3,3,Ангарск,Иркутская область,Сибирский,216973,False,Ангарск,angarsk
4,4,Архангельск,Архангельская область,Северо-Западный,294914,True,Архангельск,arkhangelsk
...,...,...,...,...,...,...,...,...
100,100,Элиста,Калмыкия,Южный,104082,True,Элиста,elista
101,101,Энгельс,Саратовская область,Приволжский,222115,False,Энгельс,engels
102,102,Южно-Сахалинск,Сахалинская область,Дальневосточный,181976,True,Южно-Сахалинск,yuzhno-sakhalinsk
103,103,Якутск,Якутия,Дальневосточный,372801,True,Якутск,yakutsk


Можно заметить, что в ссылке каждой странички города на сайте Ostrovok содержится везде одна и та же часть: 'https://ostrovok.ru/hotel/russia/' и '/?type_group=hotel.resort.boutique.apart_hotel'. Изменяется только само название. Попробуем найти все необходимые ссылки.

In [ ]:
url = 'https://ostrovok.ru/hotel/russia/'
my_cities['city_url'] = url + my_cities['name_en'] + '/?type_group=hotel.resort.boutique.apart_hotel'
print(*my_cities['city_url'], sep='\n')

https://ostrovok.ru/hotel/russia/abakan/?type_group=hotel.resort.boutique.apart_hotel
https://ostrovok.ru/hotel/russia/anadyr/?type_group=hotel.resort.boutique.apart_hotel
https://ostrovok.ru/hotel/russia/anapa/?type_group=hotel.resort.boutique.apart_hotel
https://ostrovok.ru/hotel/russia/angarsk/?type_group=hotel.resort.boutique.apart_hotel
https://ostrovok.ru/hotel/russia/arkhangelsk/?type_group=hotel.resort.boutique.apart_hotel
https://ostrovok.ru/hotel/russia/astrakhan/?type_group=hotel.resort.boutique.apart_hotel
https://ostrovok.ru/hotel/russia/balashikha/?type_group=hotel.resort.boutique.apart_hotel
https://ostrovok.ru/hotel/russia/barnaul/?type_group=hotel.resort.boutique.apart_hotel
https://ostrovok.ru/hotel/russia/belgorod/?type_group=hotel.resort.boutique.apart_hotel
https://ostrovok.ru/hotel/russia/blagoveshchensk/?type_group=hotel.resort.boutique.apart_hotel
https://ostrovok.ru/hotel/russia/blagoveshchensk/?type_group=hotel.resort.boutique.apart_hotel
https://ostrovok.ru/h

По итогам проверки всех ссылок, оказалась, что некоторые из них некорректны, поэтому исправим их и сделаем таблицу с работающими ссылками: 'итоговый список с url.xlsx'. В дальнейшим будем работать с ней.

In [ ]:
city_and_url = pd.read_excel('итоговый список с url.xlsx')
city_and_url

,Unnamed: 0,Город,Регион,Федеральный округ,Население,Наличие аэропорта,name,name_en,url
0,0,Абакан,Хакасия,Сибирский,184284,True,Абакан,abakan,https://ostrovok.ru/hotel/russia/abakan/?type_...
1,1,Анадырь,Чукотский АО,Дальневосточный,13224,True,Анадырь,anadyr,https://ostrovok.ru/hotel/russia/anadyr/?type_...
2,2,Анапа,Краснодарский край,Южный,84804,True,Анапа,anapa,https://ostrovok.ru/hotel/russia/anapa/?type_g...
3,3,Ангарск,Иркутская область,Сибирский,216973,False,Ангарск,angarsk,https://ostrovok.ru/hotel/russia/irkutsk_regio...
4,4,Архангельск,Архангельская область,Северо-Западный,294914,True,Архангельск,arkhangelsk,https://ostrovok.ru/hotel/russia/arkhangelsk/?...
...,...,...,...,...,...,...,...,...,...
99,99,Элиста,Калмыкия,Южный,104082,True,Элиста,elista,https://ostrovok.ru/hotel/russia/elista/?type_...
100,100,Энгельс,Саратовская область,Приволжский,222115,False,Энгельс,engels,https://ostrovok.ru/hotel/russia/engels/?type_...
101,101,Южно-Сахалинск,Сахалинская область,Дальневосточный,181976,True,Южно-Сахалинск,yuzhno-sakhalinsk,https://ostrovok.ru/hotel/russia/yuzhno-sakhal...
102,102,Якутск,Якутия,Дальневосточный,372801,True,Якутск,yakutsk,https://ostrovok.ru/hotel/russia/yakutsk/?type...


In [ ]:
import requests
import pandas as pd
from bs4 import BeautifulSoup
import json
import os

Логирование

In [32]:
logging.getLogger().handlers.clear()
logging.basicConfig(
    level = logging.INFO,
    filename = 'ostrovok_parser.log',
    filemode = 'w',
    format = '%(asctime)s - %(levelname)s - %(message)s',
    encoding = 'utf-8'
)

logging.info('Логирование включено')


In [33]:
with open('ostrovok_parser.log', 'r', encoding='utf-8') as f:
    print(f.read())

2026-04-07 21:19:53,929 - INFO - Логирование включено



Парсинг

In [34]:
all_cities = []
for i,j in city_and_url.iterrows():
  city_name = j['name']
  city_url = j['url']
  print(f'Название города: {city_name}')
  logging.info(f'Название города: {city_name}')
  logging.info(f'Ссылка: {city_url}')

  page = requests.get(city_url)
  soup = BeautifulSoup(page.text, 'html')

  script = soup.find('script', {'id': '__NEXT_DATA__'})
  data = json.loads(script.string)
  serp_data = data['props']['pageProps']['serpData']

  total_hotels = serp_data['totalAvailableHotels']
  page_size = serp_data['pageSize']

  total_pages = (total_hotels + page_size - 1) // page_size

  logging.info(f'{city_name}: Всего отелей {total_hotels}, Всего страниц {total_pages}')
  all_pages = []

  for page_num in range(1, total_pages+1):
    if page_num == 1:
      url_new = city_url
    else:
      url_new = city_url + f'&page={page_num}'
    logging.info(f'{city_name}: Обрабатываем страницу {page_num}')
    logging.info(f'{city_name}: {url_new}')

    page = requests.get(url_new)
    soup = BeautifulSoup(page.text, 'html')

    hotels = {}

    links = soup.find_all('a')

    for l in links:
      href = l.get('href')
      text = l.get_text()

      if href and 'mid' in href:
        if href not in hotels:
          hotels[href] = []

        if text != '':
          hotels[href].append(text)

    s = []
    for href, texts in hotels.items():
      hotel_name = None
      rating_reviews = None
      price = None

      for text in texts:
        text = text.strip()
        if 'за ночь' in text:
          price = text
        elif 'отзыв' in text:
          rating_reviews = text
        elif text != 'Показать все номера':
          hotel_name = text
      s.append({'city_name': city_name, 'hotel_url': href, 'hotel_name': hotel_name, 'rating_reviews': rating_reviews, 'price': price})

    df = pd.DataFrame(s)
    df['reviews_count'] = pd.to_numeric(df['rating_reviews'].str.extract(r'(\d+)')[0], errors='coerce')

    df['rating_value'] = pd.to_numeric(df['rating_reviews'].str.extract(r'(\d+(?:,\d+)?)$')[0].str.replace(',', '.', regex = False), errors='coerce')

    df['price_per_night'] = pd.to_numeric(df['price'].str.replace(r'\D', '', regex = True), errors='coerce')

    script = soup.find('script', {'id': '__NEXT_DATA__'})

    data = json.loads(script.string)
    hotel_json = data['props']['pageProps']['serpData']['hotels']

    s_json = []

    for hotel in hotel_json:
      s_json.append({'hotel_name_json' : hotel.get('name'),
                    'latitude': hotel.get('location', {}).get('latitude'),
                    'longitude': hotel.get('location', {}).get('longitude')})
    df2 = pd.DataFrame(s_json)
    df_merged = df.merge(df2, left_on = 'hotel_name', right_on = 'hotel_name_json', how = 'left')
    new_df = df_merged[['city_name','hotel_url','hotel_name', 'reviews_count', 'rating_value', 'price_per_night', 'latitude', 'longitude']]

    all_pages.append(new_df)
  if all_pages:
    city_df = pd.concat(all_pages, ignore_index=True)
    city_df = city_df.drop_duplicates(subset=['hotel_url'])
    city_df = city_df.reset_index(drop=True)

    all_cities.append(city_df)
    logging.info(f'{city_name}: Город успешно завершен, в итоге кол-во строк = {len(city_df)}')
  else:
    logging.warning(f'{city_name}: Нет данных по городу')

Название города: Абакан
Название города: Анадырь
Название города: Анапа
Название города: Ангарск
Название города: Архангельск
Название города: Астрахань
Название города: Балашиха
Название города: Барнаул
Название города: Белгород
Название города: Благовещенск
Название города: Братск
Название города: Брянск
Название города: Великие Луки
Название города: Владивосток
Название города: Владикавказ
Название города: Волгоград
Название города: Волжский
Название города: Вологда
Название города: Воронеж
Название города: Горно-Алтайск
Название города: Грозный
Название города: Дзержинск
Название города: Екатеринбург
Название города: Жуковский
Название города: Златоуст
Название города: Ижевск
Название города: Иркутск
Название города: Казань
Название города: Калининград
Название города: Калуга
Название города: Каменск-Уральский
Название города: Кемерово
Название города: Комсомольск-на-Амуре
Название города: Королёв
Название города: Красногорск
Название города: Краснодар
Название города: Красноярск
Н

In [37]:
with open('ostrovok_parser.log', 'r', encoding='utf-8') as f:
  l = f.readlines()
print(''.join(l[:10]))

2026-04-07 21:19:53,929 - INFO - Логирование включено
2026-04-07 21:20:02,440 - INFO - Название города: Абакан
2026-04-07 21:20:02,441 - INFO - Ссылка: https://ostrovok.ru/hotel/russia/abakan/?type_group=hotel.resort.boutique.apart_hotel
2026-04-07 21:20:03,642 - INFO - Абакан: Всего отелей 45, Всего страниц 3
2026-04-07 21:20:03,642 - INFO - Абакан: Обрабатываем страницу 1
2026-04-07 21:20:03,642 - INFO - Абакан: https://ostrovok.ru/hotel/russia/abakan/?type_group=hotel.resort.boutique.apart_hotel
2026-04-07 21:20:04,970 - INFO - Абакан: Обрабатываем страницу 2
2026-04-07 21:20:04,971 - INFO - Абакан: https://ostrovok.ru/hotel/russia/abakan/?type_group=hotel.resort.boutique.apart_hotel&page=2
2026-04-07 21:20:05,907 - INFO - Абакан: Обрабатываем страницу 3
2026-04-07 21:20:05,907 - INFO - Абакан: https://ostrovok.ru/hotel/russia/abakan/?type_group=hotel.resort.boutique.apart_hotel&page=3



Теперь соберем все данные в одну таблицу. Мы имеем следующие столбцы:

city_name - название города,

hotel_name - название отеля,

reviews_count - количество отзывов на отель,

rating_value - рейтинг отеля,

price_per_night - цена за ночь в отеле,

latitude - широта (точное расположение отеля),

longitude - долгота (точное расположение отеля).

In [38]:
city_df = pd.concat(all_cities, ignore_index=True)
city_df = city_df.drop_duplicates(subset=['hotel_url'])
city_df = city_df.reset_index(drop=True)
city_df = city_df[['city_name','hotel_name', 'reviews_count', 'rating_value', 'price_per_night', 'latitude', 'longitude']]
city_df

,city_name,hotel_name,reviews_count,rating_value,price_per_night,latitude,longitude
0,Абакан,Гостиница Хакасия,98.0,8.2,5540,53.722572,91.442980
1,Абакан,AZIMUT Отель Абакан 3*,16.0,9.3,4828,53.721160,91.439920
2,Абакан,Уютная Квартира в Центре Города,19.0,9.7,2666,53.719868,91.422040
3,Абакан,Отель Азия Бизнес,143.0,8.9,6806,53.727345,91.435680
4,Абакан,Гостиница АэроОтель,50.0,8.6,3376,53.750423,91.403275
...,...,...,...,...,...,...,...
27550,Ярославль,YAr-Sutki na ulice Uglichskaya 6,NaN,NaN,4135,57.627990,39.860070
27551,Ярославль,GoldenRing na ulice Bogdanovicha 4,NaN,NaN,4135,57.629600,39.862470
27552,Ярославль,GoldenRing na prospekte Lenina 48,NaN,NaN,4135,57.628540,39.846230
27553,Ярославль,GoldenRing na ulice Volodarskogo 50,NaN,NaN,4135,57.631508,39.865135


Сохраняем итоговую таблицу с данными.

In [39]:
city_df.to_excel('ostrovok_hotels.xlsx', index=False)